In [0]:
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:
import json

from pyspark.sql import functions as F
from common_utils.logging import get_logger
from common_utils.goldfunctions import date_key, write_gold

# ============================================================
# 1. LOGGER
# ============================================================
logger = get_logger("s2g_dimsnapshotdate")

# ============================================================
# 2. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

logger.info("Reading Gold configuration from %s", config_path)

# ============================================================
# 3. READ CONFIG
# ============================================================
with open(config_path, "r") as file:
    config = json.load(file)

# ============================================================
# 4. CONFIG SECTIONS
# ============================================================
table_config = config["dim_snapshot_date"]

source_table = table_config["source_table"]
batch_column = table_config["batch_column"]
target_table = table_config["target_table"]

# ============================================================
# 5. BUILD DIM_SNAPSHOT_DATE (one row per batch date)
# ============================================================
logger.info("Reading batch dates from %s", source_table)

dim_snapshot_date = (
    spark.table(source_table)
    .select(F.col(batch_column).alias("snapshot_date"))
    .distinct()
    .withColumn("snapshot_date_key", date_key("snapshot_date"))
    .orderBy("snapshot_date")
)


# ============================================================
# 6. DISPLAY TRANSFORMED DATA
# ============================================================
display(dim_snapshot_date)

# ============================================================
# 7. WRITE GOLD TABLE
# ============================================================
write_gold(spark, dim_snapshot_date, target_table)

# ============================================================
# 8. COMPLETED
# ============================================================
logger.info("Gold dim_snapshot_date processing completed")